# ML Explainability Agent — Playground

This notebook lets you play with everything built so far:

1. **Config-driven ingestion** — `ConfigLoader`
2. **Unified knowledge object** — `ProjectKnowledge` (model + data dictionary + context + validation)
3. **Model framework adapters** — `SklearnTreeAdapter` / `GenericModelAdapter`
4. **Intent-based routing** — `IntentClassifier`
5. **Explainability tools** — `DecisionPathExtractor`, `FeatureImportanceExtractor`

It uses the demo artifacts created by `scripts/generate_demo_artifacts.py`
(a fitted Iris decision tree, a data dictionary, a project context and a demo config).

In [ ]:
# Ensure the project root is importable and demo artifacts exist
import sys
import subprocess
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "ingestion").exists():
    ROOT = ROOT.parent  # running from notebooks/
sys.path.insert(0, str(ROOT))

DEMO_CONFIG = ROOT / "configs" / "demo_config.yaml"
if not DEMO_CONFIG.exists():
    subprocess.run(
        [sys.executable, str(ROOT / "scripts" / "generate_demo_artifacts.py")],
        check=True,
    )

print("Project root:", ROOT)
print("Demo config: ", DEMO_CONFIG)

## 1. Config-driven ingestion

`ConfigLoader` reads the demo YAML. Every artifact path comes from configuration —
no hardcoded paths.

In [ ]:
from utils.config_loader import ConfigLoader

config = ConfigLoader(str(DEMO_CONFIG))

model_path = config.get("ingestion", "model_path")
dictionary_path = config.get("ingestion", "data_dictionary_path")
context_path = config.get("ingestion", "project_context_path")

model_path, dictionary_path, context_path

## 2. Unified knowledge object

`ProjectKnowledge` composes the three loaders plus the validator behind one `build()` call.

In [ ]:
from ingestion.project_knowledge import ProjectKnowledge

builder = ProjectKnowledge(
    model_path=model_path,
    dictionary_path=dictionary_path,
    context_path=context_path,
)
knowledge = builder.build()

print("Sections:", list(knowledge.keys()))
knowledge["validation"]

## 3. Model framework adapter

The fitted decision tree is recognized by `SklearnTreeAdapter`, which adds
`framework` and `tree_details` on top of the normalized metadata.

In [ ]:
model_meta = knowledge["model"]

print("Adapter:      ", model_meta["adapter"])
print("Model type:   ", model_meta["model_type"])
print("Task type:    ", model_meta["task_type"])
print("Features:     ", model_meta["feature_names"])
print("Classes:      ", model_meta["classes"])
print("Tree details: ", model_meta.get("tree_details"))

## 4. Intent-based routing

`IntentClassifier` maps a plain-English question to a normalized intent using the
config-driven keyword map. Unknown questions fall back to the full pipeline.

In [ ]:
from tools.routing.intent_classifier import IntentClassifier

classifier = IntentClassifier(
    intents=config.get("routing", "intents", default={}),
    default_intent=config.get("routing", "default_intent", default="full_explanation"),
)

questions = [
    "What will the model predict for this flower?",
    "Why did the model choose this species?",
    "Which features are most important?",
    "Tell me something interesting.",
]

for question in questions:
    print(f"{classifier.classify(question):20s} <- {question}")

## 5. Explainability tools

Using the live model retained on the knowledge builder, extract the decision path and
feature importance for a single sample.

In [ ]:
import numpy as np

from tools.tree_reader.decision_path_extractor import DecisionPathExtractor
from tools.tree_reader.feature_importance_extractor import FeatureImportanceExtractor

model = builder.model
feature_names = model_meta["feature_names"]

sample = np.array([5.1, 3.5, 1.4, 0.2])  # a typical setosa sample

prediction = model.predict(sample.reshape(1, -1))[0]
print("Predicted class:", prediction)

path = DecisionPathExtractor(model, feature_names).extract_path(sample)
path

In [ ]:
importance = FeatureImportanceExtractor(model, feature_names).get_top_features(4)
importance

## 6. Try your own sample and question

Change the values below and re-run to explore how the routing and explanations respond.

In [ ]:
my_question = "Which features are most important for this prediction?"
my_sample = np.array([6.7, 3.0, 5.2, 2.3])  # a virginica-like sample

intent = classifier.classify(my_question)
prediction = model.predict(my_sample.reshape(1, -1))[0]

print("Question:  ", my_question)
print("Intent:    ", intent)
print("Prediction:", prediction)

if intent in ("decision_path", "full_explanation"):
    print("\nDecision path:")
    for rule in DecisionPathExtractor(model, feature_names).extract_path(my_sample)["path"]:
        print("  ", rule["condition"])

if intent in ("feature_importance", "full_explanation"):
    print("\nTop features:")
    print(FeatureImportanceExtractor(model, feature_names).get_top_features(4).to_string(index=False))